# 03 — Rat snRNA-seq validation / cerebral ischemia, GSE250245

**Source:** [GEO GSE250245](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE250245); Bormann et al., *Nature Communications* 2024.

Model: **permanent right MCAO** in male rats, brains collected **48 h**, MRI-confirmed lesion severity.

Source paper: 4 sham rats; 3 moderate stroke rats; 4 severe stroke rats. **GEO deposits six condition/hemisphere matrices**:
- Sham left/right
- Moderate MCAO contralateral/ipsilateral
- Severe MCAO contralateral/ipsilateral

**Do not assume six deposited matrices are six biological replicates.** Distinct biological animals cannot presently be identified within the released matrices; thus this is descriptive cross-species evidence only, *not* animal-replicated DE.

The rat and mouse studies differ in model, sampling, sequencing modality and time. Opposite effects are not automatically biological contradictions.

All genes screened in the mouse datasets remain frozen.

In [ ]:
!pip -q install "scanpy>=1.10,<1.12" "anndata>=0.10"

import os, sys, gzip, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.io import mmread
import scanpy as sc, anndata as ad
import requests

ROOT=Path(os.environ.get("IMACS_ROOT","/content/imacs_stroke_pet_rat" if Path("/content").exists() else "imacs_stroke_pet_rat"))
RAW=ROOT/"raw"; OUT=ROOT/"results"
RAW.mkdir(parents=True,exist_ok=True);OUT.mkdir(parents=True,exist_ok=True)
print("Python",sys.version.split()[0],"Scanpy",sc.__version__)

In [ ]:
URL="https://raw.githubusercontent.com/Touf00/retinal-nvu-computational-analysis/imacs-stroke-pet-screen/IMACS_Stroke_PET/config/candidate_catalog.csv"
CANDIDATES=pd.read_csv(URL).drop_duplicates("gene")
DESIGN=pd.DataFrame([
("GSM7976206","Sham_L","Sham","contralateral_or_left",4),
("GSM7976207","Sham_R","Sham","ipsilateral_or_right",4),
("GSM7976208","mMCAO_contra","moderate","contralateral",3),
("GSM7976209","mMCAO_ipsi","moderate","ipsilateral",3),
("GSM7976210","sMCAO_contra","severe","contralateral",4),
("GSM7976211","sMCAO_ipsi","severe","ipsilateral",4)
],columns=["gsm","prefix","severity","hemisphere","reported_group_n_rats"])
display(DESIGN)
DESIGN.to_csv(OUT/"03_rat_GEO_design_manifest.csv",index=False)

In [ ]:
def fetch(url,p):
    if p.is_file() and p.stat().st_size>0:return p
    with requests.get(url,stream=True,timeout=300) as r:
        r.raise_for_status()
        with p.open("wb") as f:
            for part in r.iter_content(1<<20):
                if part:f.write(part)
    return p

def checksum(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for part in iter(lambda:f.read(1<<20),b""):h.update(part)
    return h.hexdigest()

def load_rna(gsm,prefix):
    base=f"https://ftp.ncbi.nlm.nih.gov/geo/samples/{gsm[:-3]}nnn/{gsm}/suppl/"
    folder=RAW/gsm;folder.mkdir(exist_ok=True)
    files={}
    for ext in ("matrix.mtx.gz","features.tsv.gz","barcodes.tsv.gz"):
        name=f"{gsm}_{prefix}_{ext}"
        files[ext]=fetch(base+name,folder/name)
    with gzip.open(files["matrix.mtx.gz"],"rb") as f:
        X=mmread(f).T.tocsr()
    features=pd.read_csv(files["features.tsv.gz"],sep="\t",header=None,compression="gzip")
    genes=features.iloc[:,1].astype(str) if features.shape[1]>1 else features.iloc[:,0].astype(str)
    with gzip.open(files["barcodes.tsv.gz"],"rt") as f:
        barcodes=[line.rstrip() for line in f]
    a=ad.AnnData(X=X)
    a.var_names=pd.Index(genes.to_numpy(dtype=str),name=None)
    a.var_names_make_unique()
    a.obs_names=pd.Index([gsm+":"+x for x in barcodes])
    assert a.n_obs==len(barcodes) and a.n_vars==len(genes)
    return a,files

In [ ]:
import difflib
feature_diagnostics=[]
MARKERS={
"Endothelial":["Pecam1","Cldn5","Kdr","Emcn","Esam","Ramp2"],
"Pericyte":["Pdgfrb","Rgs5","Abcc9","Kcnj8","Cspg4"],
"VSMC":["Acta2","Tagln","Myh11","Cnn1"],
"Astrocyte":["Aqp4","Slc1a3","Aldh1l1","Gja1"],
"Microglia":["P2ry12","Tmem119","Hexb","C1qa"],
"Myeloid":["Lyz2","Ptprc","Tyrobp","Aif1"],
"Neuron":["Snap25","Rbfox3","Syt1","Map2"],
"Oligodendrocyte":["Mbp","Plp1","Mog","Mag"],
"OPC":["Pdgfra","Olig1","Olig2"],
"VLMC_Fibroblast":["Col1a1","Col1a2","Dcn","Lum"]
}

def mean_score(a,present):
    j=a.var_names.get_indexer(present)
    sub=a.X[:,j]
    return np.asarray(sub.mean(axis=1)).ravel()

def per_sample(gsm,prefix,severity,hemisphere,n):
    a,paths=load_rna(gsm,prefix)
    if gsm=="GSM7976206":
        pd.DataFrame({"rat_gene_symbol":a.var_names}).to_csv(OUT/"03_rat_reference_all_gene_symbols.csv.gz",index=False,compression="gzip")
    present=set(a.var_names)
    for g in CANDIDATES.gene:
        if g not in present:
            nearby=[x for x in a.var_names if g.lower() in x.lower() or x.lower() in g.lower()]
            matches=difflib.get_close_matches(g,list(a.var_names),n=5,cutoff=.65)
            feature_diagnostics.append({"gsm":gsm,"gene":g,"not_found":True,
                                        "case_insensitive_or_substring":",".join(nearby[:10]),
                                        "fuzzy_matches":",".join(matches)})
    a.var["mt"]=a.var_names.str.upper().str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a,qc_vars=["mt"],inplace=True)
    before=a.n_obs
    passing=(a.obs["n_genes_by_counts"]>=200)&(a.obs["pct_counts_mt"]<20)
    passing&=a.obs["n_genes_by_counts"]<=a.obs["n_genes_by_counts"].quantile(.995)
    passing&=a.obs["total_counts"]<=a.obs["total_counts"].quantile(.995)
    a=a[passing].copy()
    sc.pp.normalize_total(a,target_sum=1e4)
    sc.pp.log1p(a)
    scores={}
    for ct,genes in MARKERS.items():
        present=[x for x in genes if x in a.var_names]
        if len(present)>=2:
            scores[ct]=mean_score(a,present)
    matrix=pd.DataFrame(scores,index=a.obs_names)
    ordered=np.argsort(matrix.values,axis=1)
    rank=np.sort(matrix.values,axis=1)
    lead=np.array(matrix.columns)[ordered[:,-1]]
    margin=rank[:,-1]-rank[:,-2]
    label=np.where((margin>.15)&(rank[:,-1]>.15),lead,"Ambiguous")
    a.obs["celltype"]=label
    comp=pd.Series(label).value_counts()
    print(gsm,severity,hemisphere,":",before,"->",a.n_obs,"cells; recovered:",comp.to_dict())
    records=[]
    for ct in sorted(set(label)):
        ix=np.flatnonzero(label==ct)
        for gene in CANDIDATES.gene:
            if gene not in a.var_names:
                records.append(dict(gsm=gsm,prefix=prefix,severity=severity,hemisphere=hemisphere,
                  celltype=ct,gene=gene,n_cells=len(ix),mean_log1p=np.nan,detection_fraction=np.nan))
                continue
            j=a.var_names.get_loc(gene)
            # Sparse log-normalized counts: zero remains exactly zero
            t=a.X[ix,j]
            t=np.asarray(t.toarray()).ravel() if sp.issparse(t) else np.asarray(t).ravel()
            records.append(dict(gsm=gsm,prefix=prefix,severity=severity,hemisphere=hemisphere,
              celltype=ct,gene=gene,n_cells=len(ix),mean_log1p=float(t.mean()),
              detection_fraction=float((t>0).mean())))
    qc=dict(gsm=gsm,severity=severity,hemisphere=hemisphere,n_cells_before=before,n_cells_after=a.n_obs,
            median_genes_after=float(a.obs["n_genes_by_counts"].median()),
            median_pct_mt_after=float(a.obs["pct_counts_mt"].median()))
    digest=[dict(gsm=gsm,file=p.name,bytes=p.stat().st_size,sha256=checksum(p)) for p in paths.values()]
    return pd.DataFrame(records),qc,digest

In [ ]:
all_expr=[];qc=[];provenance=[]
for row in DESIGN.itertuples(index=False):
    print("Processing:",row.gsm,row.prefix)
    e,q,h=per_sample(row.gsm,row.prefix,row.severity,row.hemisphere,row.reported_group_n_rats)
    all_expr.append(e);qc.append(q);provenance.extend(h)
    # Clear large count matrices between groups
    import gc
    gc.collect()

E=pd.concat(all_expr,ignore_index=True)
E.to_csv(OUT/"03_rat_48h_candidate_expression_by_group_and_hemisphere.csv",index=False)
pd.DataFrame(feature_diagnostics,columns=["gsm","gene","not_found",
    "case_insensitive_or_substring","fuzzy_matches"]).to_csv(OUT/"03_rat_missing_gene_symbol_audit.csv",index=False)
pd.DataFrame(qc).to_csv(OUT/"03_rat_48h_QC_audit.csv",index=False)
pd.DataFrame(provenance).to_csv(OUT/"03_rat_SHA256.csv",index=False)
display(pd.DataFrame(qc))
print("Saved",len(E),"gene-by-compartment observations")

In [ ]:
from itertools import product
rows=[]
comparisons=[
("mMCAO_ipsi","Sham_R"),("sMCAO_ipsi","Sham_R"),
("mMCAO_ipsi","mMCAO_contra"),("sMCAO_ipsi","sMCAO_contra"),
]
for (case,control), (ct,g) in product(comparisons,E.groupby(["celltype","gene"],dropna=False)):
    x=g.set_index("prefix")
    if case not in x.index or control not in x.index:continue
    ca=x.loc[case];co=x.loc[control]
    rows.append(dict(celltype=ct[0],gene=ct[1],case=case,control=control,
      log_mean_difference=float(ca.mean_log1p-co.mean_log1p),
      detection_fraction_difference=float(ca.detection_fraction-co.detection_fraction),
      case_n_cells=int(ca.n_cells),control_n_cells=int(co.n_cells),
      inference="DESCRIPTIVE ONLY—six pooled condition/hemisphere matrices, no animal-level replication"))
C=pd.DataFrame(rows)
C["adequate_cell_capture"]= (C["case_n_cells"]>=50)&(C["control_n_cells"]>=50)
C["comparison_qc"]=np.where(C["adequate_cell_capture"],
    "PASS_COUNTS_ONLY_DESCRIPTIVE","FAIL_LOW_CELL_COUNTS_DO_NOT_INTERPRET")
C.to_csv(OUT/"03_rat_group_comparisons_descriptive_NO_PVALUES.csv",index=False)
display(C[C.celltype.isin(["Endothelial","Astrocyte","Pericyte"])].head(100))

In [ ]:
import json
(OUT/"03_rat_run_manifest.json").write_text(json.dumps({
 "dataset":"GSE250245","n_deposited_group_hemisphere_matrices":6,
 "design":"4 sham, 3 moderate MCAO, 4 severe MCAO; ipsilateral/contralateral",
 "biological_replication_for_analysis":"NOT IDENTIFIABLE FROM DEPOSITED GROUP MATRICES",
 "inference":"DESCRIPTIVE ONLY; no cell-level inferential statistics",
 "key_caveat":"snRNA at 48h permanent MCAO differs from mouse scRNA at 24h transient MCAO"
},indent=2))
print("FINAL OUTPUTS",sorted(q.name for q in OUT.iterdir()))

## Interpretation checklist

- Compare rat cortex-wide *nuclear* RNA with mouse tissue-derived *cellular* RNA descriptively.
- No `padj`, `pvalue`, or false claim of n=11 independent single-nucleus matrices.
- Lesion severity is a property of a condition-level matrix; lack of within-condition dispersion prevents inference.
- Endothelial loss, abundance and gene expression can be affected by tissue dissection and nuclear capture.
- Do not conclude receptor transport function or target cell surface accessibility from RNA.
- Whether animal-level data can be reconstructed depends on deposited barcode/source metadata and must be audited separately.
